# ⚽ CanchaVision en GPU (Google Colab)

Corre el motor de análisis en una **GPU gratuita** de Colab. Lo que en un portátil sin GPU tarda ~70 min, aquí debería tardar unos **2–4 min**.

**Antes de empezar:** menú `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU (T4)**.

Luego ejecuta las celdas en orden (▶️).

## 1. Comprobar que hay GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo
Edita `REPO_URL` con la URL de tu repositorio de GitHub.

In [ ]:
REPO_URL = "https://github.com/TU_USUARIO/canchavision.git"  # <-- EDITA ESTO
!git clone $REPO_URL
%cd canchavision

## 3. Instalar dependencias (versión GPU)
Usa `inference-gpu` (onnxruntime en GPU). Tarda un par de minutos.

In [ ]:
!pip install -q -e .
!pip install -q inference-gpu "git+https://github.com/roboflow/sports.git" gdown

## 4. Tu API key de Roboflow
Se pide de forma oculta (no queda guardada en el notebook).

In [ ]:
import os, getpass
os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")

## 5. Descargar un clip de prueba

In [ ]:
!mkdir -p data/raw
!gdown -O data/raw/2e57b9_0.mp4 "https://drive.google.com/uc?id=19PGw55V8aA6GZu5-Aac5_9mCy3fNxmEf"

## 6. Procesar en GPU (y medir el tiempo)
Compara este tiempo con los ~70 min del portátil sin GPU.

In [ ]:
import time
t = time.time()
!canchavision --video data/raw/2e57b9_0.mp4 --config config/football_roboflow.yaml --max-frames 300
print(f"\n⏱️ Tiempo total en GPU: {time.time() - t:.0f} s")

## 7. Ver resultados: pizarrón cenital y mapa de calor

In [ ]:
import cv2
from IPython.display import Image, display
cap = cv2.VideoCapture('outputs/2e57b9_0_radar.mp4')
cap.set(cv2.CAP_PROP_POS_FRAMES, 150)
ok, f = cap.read()
cv2.imwrite('outputs/_radar_frame.png', f)
print('Pizarrón cenital:'); display(Image('outputs/_radar_frame.png'))
print('Mapa de calor:'); display(Image('outputs/2e57b9_0_heatmap.png'))

## 8. Stats por jugador, posesión y pases

In [ ]:
!python scripts/compute_stats.py outputs/2e57b9_0_positions.csv 25
print('\n' + '='*40)
!python scripts/compute_possession.py outputs/2e57b9_0_positions.csv outputs/2e57b9_0_ball.csv